In [ ]:
import pandas as pd, glob, os
from pathlib import Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:


# === CONFIG ===
BASE = "/content/drive/MyDrive/PropInsight/labeled"   # main labeled folder
pattern = f"{BASE}/**/*_processed_labeled_enriched.csv"        # recursively find all labeled CSVs
files = glob.glob(pattern, recursive=True)

print(f"Found {len(files)} labeled files:\n")
for f in files: print(" -", f)
print("\n")

Found 15 labeled files:

 - /content/drive/MyDrive/PropInsight/labeled/government_websites/bca/bca_media_releases_20250921_173345_processed_labeled_enriched.csv
 - /content/drive/MyDrive/PropInsight/labeled/government_websites/hdb/hdb_articles_20250921_165216_processed_labeled_enriched.csv
 - /content/drive/MyDrive/PropInsight/labeled/government_websites/mas/mas_banking_regulation_20250921_172001_processed_labeled_enriched.csv
 - /content/drive/MyDrive/PropInsight/labeled/government_websites/mas/mas_macroprudential_policy_20250921_172001_processed_labeled_enriched.csv
 - /content/drive/MyDrive/PropInsight/labeled/government_websites/mas/mas_articles_20250921_172001_processed_labeled_enriched.csv
 - /content/drive/MyDrive/PropInsight/labeled/government_websites/mas/mas_parliamentary_speech_20250921_172001_processed_labeled_enriched.csv
 - /content/drive/MyDrive/PropInsight/labeled/government_websites/mnd/mnd_articles_20250921_164356_processed_labeled_enriched.csv
 - /content/drive/MyDri

In [ ]:


# === Expected PropInsight label fields ===
required_fields = [
    # Core sentiment
    "overall_sentiment", "price_sentiment", "policy_sentiment",
    "affordability_sentiment", "location_sentiment", "lifestyle_sentiment",
    # Metadata
    "date", "temporal_bucket", "thread_url", "forum_name",
    # Entities / Aspects
    "aspect", "entity", "prop_topic_tags", "entity_labels",
    # Location
    "location", "locations",
    # Singlish / Cultural
    "has_singlish", "singlish_terms", "singlish_meanings",
    # Policy
    "rx_ABSD", "rx_TDSR", "flag_ABSD", "flag_TDSR",
]

# === Helper to print short lists nicely ===
def short(lst, n=8):
    if len(lst) <= n: return lst
    return lst[:n] + ["..."]

# === Scan each file ===
summary = []
for path in sorted(files):
    name = Path(path).stem
    print(f"\n🗂️ === {name} ===")
    try:
        df = pd.read_csv(path)
        cols = df.columns.tolist()
        present = [c for c in required_fields if c in cols]
        missing = [c for c in required_fields if c not in cols]
        print(f"Rows: {len(df)} | Columns: {len(cols)}")
        print(f"✅ Present ({len(present)}):", short(present))
        print(f"⚠️ Missing ({len(missing)}):", short(missing))

        # Quick feature presence summary
        flags = {
            "Sentiment": any("sentiment" in c for c in cols),
            "Aspect": any("aspect" in c for c in cols),
            "Entity": any("entity" in c for c in cols),
            "Location": any("location" in c for c in cols),
            "Date": any("date" in c for c in cols),
            "Singlish": any("singlish" in c for c in cols),
            "Policy": any("absd" in c.lower() or "tdsr" in c.lower() for c in cols),
        }
        print("→ Flags:", flags)

        # Sentiment value preview
        sentiment_cols = [c for c in cols if "sentiment" in c]
        if sentiment_cols:
            print("🎯 Sentiment columns:", sentiment_cols)
            for c in sentiment_cols:
                vc = df[c].value_counts(dropna=False).head(5).to_dict()
                print(f"  {c}: {vc}")

        summary.append({
            "file": name,
            "rows": len(df),
            **{f.lower(): v for f, v in flags.items()},
            "present": len(present),
            "missing": len(missing)
        })
        print("-" * 90)
    except Exception as e:
        print(f"❌ Error reading {path}: {e}")
        print("-" * 90)

# === Optional: Overall Summary Table ===
summary_df = pd.DataFrame(summary)
if not summary_df.empty:
    print("\n\n=== 🔎 OVERALL COVERAGE SUMMARY ===")
    display(summary_df)
else:
    print("No files analyzed.")



🗂️ === bca_media_releases_20250921_173345_processed_labeled_enriched ===
Rows: 7 | Columns: 40
✅ Present (7): ['overall_sentiment', 'price_sentiment', 'policy_sentiment', 'affordability_sentiment', 'location_sentiment', 'date', 'temporal_bucket']
⚠️ Missing (16): ['lifestyle_sentiment', 'thread_url', 'forum_name', 'aspect', 'entity', 'prop_topic_tags', 'entity_labels', 'location', '...']
→ Flags: {'Sentiment': True, 'Aspect': True, 'Entity': False, 'Location': True, 'Date': True, 'Singlish': True, 'Policy': False}
🎯 Sentiment columns: ['property_sentiment', 'aspect_based_sentiment', 'overall_sentiment', 'price_sentiment', 'investor_sentiment', 'sentiment_intensity', 'sentiment_drivers', 'policy_sentiment', 'supply_sentiment', 'economic_sentiment', 'demand_sentiment', 'infrastructure_sentiment', 'affordability_sentiment', 'location_sentiment']
  property_sentiment: {'{"overall_sentiment": {"value": "positive", "confidence_score": 0.85}, "property_market_sentiment": {"value": "bullish",

,file,rows,sentiment,aspect,entity,location,date,singlish,policy,present,missing
0,bca_media_releases_20250921_173345_processed_l...,7,True,True,False,True,True,True,False,7,16
1,hdb_articles_20250921_165216_processed_labeled...,32,True,True,False,True,True,True,False,7,16
2,mas_articles_20250921_172001_processed_labeled...,23,True,True,False,True,True,True,False,7,16
3,mas_banking_regulation_20250921_172001_process...,18,True,True,False,True,True,True,False,7,16
4,mas_macroprudential_policy_20250921_172001_pro...,1,True,True,False,True,True,True,False,7,16
5,mas_parliamentary_speech_20250921_172001_proce...,4,True,True,False,True,True,True,False,7,16
6,mnd_articles_20250921_164356_processed_labeled...,5,True,True,False,True,True,True,False,7,16
7,sfa_agricultural_land_20250921_174426_processe...,3,True,True,False,True,True,True,False,7,16
8,sfa_articles_20250921_174426_processed_labeled...,28,True,True,False,True,True,True,False,7,16
9,sfa_food_retail_20250921_174426_processed_labe...,3,True,True,False,True,True,True,False,7,16
